# 01 - Análise Exploratória de Dados (EDA)

## Passos Mágicos - Predição de Risco de Defasagem Escolar

Este notebook realiza a análise exploratória dos dados da Associação Passos Mágicos.

### Objetivos:
1. Entender a estrutura dos dados
2. Analisar distribuições dos indicadores
3. Identificar padrões e correlações
4. Detectar valores ausentes e outliers

In [ ]:
# Imports
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.preprocessing import DataLoader
from src.config import INDICADORES, PEDRA_LIMITES

# Configurações
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 50)
%matplotlib inline

## 1. Carregamento dos Dados

In [ ]:
# Carregar dados
loader = DataLoader()
df = loader.load_and_unify()

print(f"Shape: {df.shape}")
print(f"\nColunas: {list(df.columns)}")

In [ ]:
# Informações gerais
df.info()

In [ ]:
# Primeiras linhas
df.head()

## 2. Análise de Valores Ausentes

In [ ]:
# Percentual de valores ausentes
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    'Coluna': missing.index,
    'Faltantes': missing.values,
    'Percentual (%)': missing_pct.values
}).sort_values('Faltantes', ascending=False)

missing_df[missing_df['Faltantes'] > 0]

In [ ]:
# Visualização de missing values
fig, ax = plt.subplots(figsize=(12, 6))
missing_cols = missing_df[missing_df['Faltantes'] > 0]['Coluna'].head(15)
missing_vals = missing_df[missing_df['Faltantes'] > 0]['Percentual (%)'].head(15)

sns.barplot(x=missing_vals.values, y=missing_cols.values, palette='Reds_r')
plt.xlabel('Percentual de Valores Ausentes (%)')
plt.title('Top 15 Colunas com Valores Ausentes')
plt.tight_layout()
plt.show()

## 3. Análise dos Indicadores

In [ ]:
# Estatísticas descritivas dos indicadores
indicador_cols = [col for col in df.columns if col in ['INDE', 'IAN', 'IDA', 'IEG', 'IAA', 'IPS', 'IPP', 'IPV']]
df[indicador_cols].describe()

In [ ]:
# Distribuição dos indicadores
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(indicador_cols[:8]):
    if col in df.columns:
        sns.histplot(df[col].dropna(), kde=True, ax=axes[i], color='steelblue')
        axes[i].set_title(f'{col}\n{INDICADORES.get(col, "")}')
        axes[i].axvline(df[col].mean(), color='red', linestyle='--', label='Média')
        axes[i].legend()

plt.tight_layout()
plt.suptitle('Distribuição dos Indicadores Educacionais', y=1.02, fontsize=14)
plt.show()

## 4. Análise de Correlações

In [ ]:
# Matriz de correlação dos indicadores
corr_matrix = df[indicador_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='RdBu_r', center=0, 
            square=True, fmt='.2f', linewidths=0.5)
plt.title('Matriz de Correlação - Indicadores')
plt.tight_layout()
plt.show()

## 5. Análise por Classificação PEDRA

In [ ]:
# Distribuição por PEDRA
if 'PEDRA' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Contagem
    pedra_counts = df['PEDRA'].value_counts()
    colors = ['#E0E0E0', '#9370DB', '#8B008B', '#FFD700']
    pedra_counts.plot(kind='bar', ax=axes[0], color=colors[:len(pedra_counts)])
    axes[0].set_title('Distribuição por Classificação PEDRA')
    axes[0].set_ylabel('Quantidade')
    
    # INDE por PEDRA
    sns.boxplot(data=df, x='PEDRA', y='INDE', ax=axes[1], palette=colors[:len(pedra_counts)])
    axes[1].set_title('INDE por Classificação PEDRA')
    
    plt.tight_layout()
    plt.show()

## 6. Análise Temporal

In [ ]:
# Evolução por ano PEDE
if 'ANO_PEDE' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Quantidade por ano
    df.groupby('ANO_PEDE').size().plot(kind='bar', ax=axes[0], color='steelblue')
    axes[0].set_title('Quantidade de Registros por Ano')
    axes[0].set_ylabel('Quantidade')
    
    # INDE médio por ano
    df.groupby('ANO_PEDE')['INDE'].mean().plot(kind='bar', ax=axes[1], color='green')
    axes[1].set_title('INDE Médio por Ano')
    axes[1].set_ylabel('INDE Médio')
    
    plt.tight_layout()
    plt.show()

## 7. Conclusões da EDA

### Principais Descobertas:
1. **Estrutura dos dados**: Dados de múltiplos anos (2022-2024) com indicadores educacionais
2. **Valores ausentes**: Identificados e tratados adequadamente
3. **Distribuições**: Indicadores seguem distribuições aproximadamente normais
4. **Correlações**: Forte correlação entre indicadores acadêmicos
5. **PEDRA**: Classificação bem definida pelos ranges de INDE

### Próximos Passos:
- Feature Engineering
- Criação da variável target (RISCO_DEFASAGEM)
- Seleção de features